# Baseline notebook

Create model baseline and analysis for it

In [ ]:
import pandas as pd 
import numpy as np

pd.set_option('display.max_columns', 200)
np.random.seed(42)

In [ ]:
%pip install -q kagglehub

In [ ]:
# download dataset from kaggle
import kagglehub # pyright: ignore[reportMissingImports]
from pathlib import Path

# Download latest version
path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
path = path / r"WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(path)
df.head()

## Data preparation (From EDA)

In [ ]:
to_category_columns = (
    [
        "gender",
        "Partner",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup", 
        "DeviceProtection",
        "TechSupport",
        "StreamingTV", 
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod",
        "Churn"
        ])
    
for column in to_category_columns:
    df[column] = df[column].astype("category")
    
df["SeniorCitizen"] = df["SeniorCitizen"] == 1
#FIXME: take a look later
df["Churn"] = df["Churn"] == "Yes"
    
df["TotalCharges"]  = pd.to_numeric(df['TotalCharges'], errors='coerce')

df = df.dropna()
df["TotalCharges"].isna().sum()

print("Data preparation is done")
df.shape

## Model building

In [ ]:
df.columns

In [ ]:
# Drop customerId column as never significant
df = df.drop(columns="customerID")
df.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder # type: ignore

df_ohe = pd.get_dummies(df)
df_ohe.shape

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

X = df_ohe.drop(columns="Churn").copy()
y = df_ohe["Churn"].copy()

rf = RandomForestClassifier(random_state=42)

y_pred_cv = cross_val_predict(rf, X, y, cv=5)
print(classification_report(y, y_pred_cv))

## Metrics explained

It simpler to explain on one class. I will use where Churn is True

Class True (Churn is True):
- Precision 0.64 — when model predicts "churn", it's correct only 64% of the time
- Recall 0.48 — it catches only 48% of actual churners

Same logic for the not churned users

As we are trying to minimize missed by model churns (FN error that costs business big money). We should maximize recall on True class.

Despite model performs well on false classes (can classify not churning customers) it's perform nearly a coin flip predict on actually churned ones. Such conduct can be caused by class imbalance (roughly $\approx30%$% customers from data were churned)



What to improve:
- Handle class imbalance (class_weight='balanced' or SMOTE)
- Try other models (XGBoost, Logistic Regression with tuning)
- Use StratifiedKFold for cross-validation
- Focus on recall for class True as the main metric (missing churners is costly)